In [1]:
from utils.analysis.valuation import BuySellSignalsAnalyzer, SignalsReporter
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE, TRADING_SIGNALS_CONFIG

In [2]:
# 📊 CONFIGURACIÓN
# Las fechas vienen de config.py (TRADING_SIGNALS_CONFIG)
# Personaliza aquí solo si necesitas valores diferentes

# Tickers a analizar
TICKERS  = ["SYF", "CF", "AFL", "NEM"]   # Agrega más: ["AAPL", "MSFT", "GOOGL", ...]

# Lookback para análisis de precios
LOOKBACK_YEARS = 5  # Años de histórico

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
CUSTOM_START = ""  # Ej: "2022-01-01"
CUSTOM_END = ""    # Ej: "2024-12-31"

print("📋 Configuración:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Lookback: {LOOKBACK_YEARS} años")
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    print(f"   Fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    print(f"   Fechas desde config.py: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE}")

📋 Configuración:
   Tickers: 4
   Lookback: 5 años
   Fechas desde config.py: 2020-01-01 → 2025-12-24


In [3]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager
data_manager = DataManager()

# Crear analyzer con configuración
# Si USE_CUSTOM_DATES es True, usar fechas personalizadas
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        start_date=CUSTOM_START,
        end_date=CUSTOM_END,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con configuración de config.py")

reporter = SignalsReporter()
print("✅ Reporter inicializado")

✅ Analyzer con configuración de config.py
✅ Reporter inicializado


In [4]:
# 🔍 ANÁLISIS DE SEÑALES
signals = []

print(f"\n🔍 Analizando {len(TICKERS)} ticker(s)...\n")

for ticker in TICKERS:
    try:
        # El analyzer ya tiene las fechas configuradas
        signal = analyzer.analyze_stock(ticker)
        signals.append(signal)
        
        # Mostrar resumen rápido
        emoji = "🟢" if signal.signal == "COMPRA" else "🔴" if signal.signal == "VENTA" else "🟡"
        print(f"{emoji} {ticker}: {signal.signal} "
              f"(Conf: {signal.confidence:.1f}%, Pot: {signal.upside_potential:+.1f}%)")
        
    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)[:60]}")

print(f"\n📊 Total analizado: {len(signals)}/{len(TICKERS)} tickers")


🔍 Analizando 4 ticker(s)...

Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


🟡 SYF: MANTENER (Conf: 50.0%, Pot: +0.7%)
Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


🟢 CF: COMPRA (Conf: 77.9%, Pot: +17.4%)
Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed


🟢 AFL: COMPRA (Conf: 74.0%, Pot: +0.1%)
Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed

🟢 NEM: COMPRA (Conf: 71.3%, Pot: +3.5%)

📊 Total analizado: 4/4 tickers


In [5]:
# 📊 RESUMEN EN TABLA
if signals:
    summary_df = reporter.to_dataframe(signals)
    display(summary_df)
    
    print()
    reporter.print_summary(signals)
else:
    print("❌ No hay señales para mostrar")

,Ticker,Señal,Confianza,Valoración,Fundamental,Precio Actual,Precio Objetivo,Potencial
0,SYF,MANTENER,50.0%,89.9,nan,$86,$86,0.7%
1,CF,COMPRA,77.9%,89.5,75.4,$78,$92,17.4%
2,AFL,COMPRA,74.0%,80.1,74.9,$111,$111,0.1%
3,NEM,COMPRA,71.3%,71.2,79.2,$105,$109,3.5%




RESUMEN DE SEÑALES

🟢 COMPRAS: 3
   CF: 77.9% confianza, 17.4% potencial
   AFL: 74.0% confianza, 0.1% potencial
   NEM: 71.3% confianza, 3.5% potencial

🔴 VENTAS: 0

🟡 MANTENER: 1


In [6]:
# 🏆 TOP OPORTUNIDADES DE COMPRA
if signals:
    reporter.print_top_opportunities(signals, top_n=5)
else:
    print("❌ No hay señales para analizar")


TOP 5 OPORTUNIDADES DE COMPRA

1. CF
   Confianza: 77.9%
   Potencial: 17.4%
   Precio: $78 → $92
   Valoración: 89.5 | Fundamental: 75.4
   Razones: 💰 Valoración atractiva - Precio por debajo del valor justo, ✅ Sólidos fundamentales

2. AFL
   Confianza: 74.0%
   Potencial: 0.1%
   Precio: $111 → $111
   Valoración: 80.1 | Fundamental: 74.9
   Razones: 💰 Valoración atractiva - Precio por debajo del valor justo, ✅ Sólidos fundamentales

3. NEM
   Confianza: 71.3%
   Potencial: 3.5%
   Precio: $105 → $109
   Valoración: 71.2 | Fundamental: 79.2
   Razones: 💰 Valoración atractiva - Precio por debajo del valor justo, ✅ Sólidos fundamentales


In [9]:
# 🟢 DETALLE DE SEÑALES DE COMPRA
compras = [s for s in signals if s.signal == "COMPRA"]

if compras:
    # Ordenar por confianza
    top_compras = sorted(compras, key=lambda x: x.confidence, reverse=True)
    
    print(f"📈 Encontradas {len(compras)} señales de COMPRA\n")
    print("=" * 65)
    
    for signal in top_compras:
        reporter.print_signal(signal)
        print()
else:
    print("ℹ️  No hay señales de COMPRA en este momento")

📈 Encontradas 3 señales de COMPRA

SEÑAL DE INVERSIÓN: CF

🟢 COMPRA (Confianza: 77.9%)
📊 SCORES:
   Valoración:    🟢 [█████████████████░░░] 89.5
   Fundamental:   🟡 [███████████████░░░░░] 75.4
💰 PRECIOS:
   Actual:        $78
   Objetivo:      $92
   Potencial:     17.4%

💡 RAZONES:
   💰 Valoración atractiva - Precio por debajo del valor justo
   ✅ Sólidos fundamentales
   🚀 Alto crecimiento sostenido

SEÑAL DE INVERSIÓN: AFL

🟢 COMPRA (Confianza: 74.0%)
📊 SCORES:
   Valoración:    🟢 [████████████████░░░░] 80.1
   Fundamental:   🟡 [██████████████░░░░░░] 74.9
💰 PRECIOS:
   Actual:        $111
   Objetivo:      $111
   Potencial:     0.1%

💡 RAZONES:
   💰 Valoración atractiva - Precio por debajo del valor justo
   ✅ Sólidos fundamentales
   🚀 Alto crecimiento sostenido

SEÑAL DE INVERSIÓN: NEM

🟢 COMPRA (Confianza: 71.3%)
📊 SCORES:
   Valoración:    🟡 [██████████████░░░░░░] 71.2
   Fundamental:   🟡 [███████████████░░░░░] 79.2
💰 PRECIOS:
   Actual:        $105
   Objetivo:      $109
   Po

In [11]:
# 🏭 ANÁLISIS POR SECTOR
# Analiza múltiples tickers de un sector específico

TECH_SECTOR  = ["SYF", "CF", "AFL", "NEM"]

print("🔍 Analizando sector tecnológico...\n")
tech_signals = []

for ticker in TECH_SECTOR:
    try:
        signal = analyzer.analyze_stock(ticker)
        tech_signals.append(signal)
    except Exception as e:
        print(f"⚠️  {ticker}: {str(e)[:40]}")

if tech_signals:
    # Mostrar solo las mejores oportunidades
    compras_tech = [s for s in tech_signals if s.signal == "COMPRA"]
    
    if compras_tech:
        print(f"\n✅ {len(compras_tech)} oportunidades de compra en Tech:\n")
        
        tech_df = reporter.to_dataframe(compras_tech)
        display(tech_df.sort_values('Confianza', ascending=False))
    else:
        print("\nℹ️  No hay señales de compra en el sector tech actualmente")

🔍 Analizando sector tecnológico...



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2025-12-24



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2025-12-24



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2025-12-24



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2025-12-24

✅ 3 oportunidades de compra en Tech:



,Ticker,Señal,Confianza,Valoración,Fundamental,Precio Actual,Precio Objetivo,Potencial
0,CF,COMPRA,77.9%,89.5,75.4,$78,$92,17.4%
1,AFL,COMPRA,74.0%,80.1,74.9,$111,$111,0.1%
2,NEM,COMPRA,71.3%,71.2,79.2,$105,$109,3.5%
